In [1]:
import sys
import os
import torch
sys.path.append(os.path.abspath('../'))

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from torch_geometric.data import Data
from tqdm import tqdm

# Add both project root and src/ to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))  # <-- add this

from src.dataset import deterministic_sample, T5000Dataset, create_dataloader
from src.egnn import EGNN
from src.fm import FlowMatching
from src.pbc_config import wrap, min_image, BOX
from src.utils import load_config, gpu_knn_graph_pbc_batch, scale_thetas, SinusoidalTimeEmbedding, SinusoidalThetaEmbedding

from src.validation import get_tpcf

import numpy as np
import pandas as pd 
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import linear_sum_assignment


/home/bartb/venvs/boids/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_kwargs = {'batch_size': 16}
test_kwargs = {'batch_size': 16}

sample = deterministic_sample(
    "../Data",
    "../Data"
        )

class SingleSampleDataset(TorchDataset):
    def __init__(self, sample, size=256):
        self.sample = sample
        self.size = size
    def __len__(self): return self.size
    def __getitem__(self, idx): return self.sample

overfitting_dataset = SingleSampleDataset(sample, size=256)
train_sampler = torch.utils.data.distributed.DistributedSampler(overfitting_dataset, shuffle=False)
train_dl = DataLoader(overfitting_dataset, sampler=train_sampler, drop_last=True, **train_kwargs)
valid_dl = DataLoader(overfitting_dataset, drop_last=True, **test_kwargs)

TypeError: T5000Dataset.__init__() missing 2 required positional arguments: 'coupling' and 'optimal_transport'

In [ ]:
cosmologies_dir = "../../Data"
cosmologies_info_dir = "../../Data"

train_dataset = T5000Dataset(cosmologies_dir, cosmologies_info_dir, coupling=False, optimal_transport=False, split='train')
test_dataset = T5000Dataset(cosmologies_dir, cosmologies_info_dir, coupling=False, optimal_transport=False, split='test')

print(len(test_dataset))

In [ ]:
train_kwargs = {"batch_size": 16}
valid_kwargs = {"batch_size": 1}

train_loader, test_loader, sampler = create_dataloader(cosmologies_dir, 
                                                 
                                                       cosmologies_info_dir, 
                                                       train_kwargs=train_kwargs,
                                                       valid_kwargs=valid_kwargs,
                                                       coupling=False,
                                                       optimal_transport=False,
                                                       rotations=True,
                                                       translations=True,
                                                       distributed=False
                                                       )
graph = next(iter(train_loader))
graph.mass.shape

In [ ]:
def calc_xt(x0, x1, t):
    delta = min_image(x1 - x0, **BOX)
    mu_t = x0+t*delta
    sigma_t = 0.
    x = mu_t + sigma_t * torch.randn_like(x1)
    return wrap(x, **BOX)

def conditional_vector_field(x0, x1):
    u_t = x1 - x0
    u_t_pbc = min_image(u_t, **BOX)
    return u_t_pbc

In [ ]:
idx = np.random.choice(5000, 10, replace=False).tolist()

x1 = next(iter(train_loader)).x.view(1, 5000, 3)[:, idx, :]
x0 = torch.rand(1, 5000, 3)[:, idx, :]

t = torch.tensor(0.5)
x_t = calc_xt(x0, x1, t)
u_t = conditional_vector_field(x0, x1)

# Convert to numpy
x0_np = x0.reshape(-1, 3).cpu().numpy()
x1_np = x1.reshape(-1, 3).cpu().numpy()
x_t_np = x_t.reshape(-1, 3).cpu().numpy()
u_t_np = u_t.reshape(-1, 3).cpu().numpy()
t_val = t.item()

In [ ]:
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(111, projection='3d')
box_size = 1.0
colors = cm.tab10(np.linspace(0, 1, 10))

for j in range(10):
    p0 = x0_np[j]
    p1 = x1_np[j]
    pt = x_t_np[j]
    c = colors[j]
    
    steps = np.linspace(0, 1, 100)
    delta = p1 - p0
    delta = delta - np.round(delta / box_size) * box_size
    
    for k in range(len(steps) - 1):
        s0 = (p0 + steps[k] * delta) % box_size
        s1 = (p0 + steps[k+1] * delta) % box_size
        if np.all(np.abs(s1 - s0) < 0.5 * box_size):
            ax.plot([s0[0], s1[0]], [s0[1], s1[1]], [s0[2], s1[2]], c=c, alpha=0.4, linewidth=2)
    
    ax.scatter(p0[0], p0[1], p0[2], c=[c], s=100, marker='o', edgecolors='black', linewidths=0.5)
    ax.scatter(pt[0], pt[1], pt[2], c=[c], s=150, marker='x', linewidths=2)
    ax.scatter(p1[0], p1[1], p1[2], c=[c], s=100, marker='^', edgecolors='black', linewidths=0.5)

ax.set_xlim(0, box_size)
ax.set_ylim(0, box_size)
ax.set_zlim(0, box_size)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.scatter([], [], [], c='gray', marker='o', s=60, label='x0 (noise)')
ax.scatter([], [], [], c='gray', marker='x', s=60, label='x_t')
ax.scatter([], [], [], c='gray', marker='^', s=60, label='x1 (data)')
ax.legend()
plt.title(f't = {t_val:.2f}')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
plt.subplots_adjust(wspace=0, hspace=0)
box_size = 1.0
colors = cm.tab10(np.linspace(0, 1, 10))

for row in range(3):
    for col in range(3):
        ax = axes[row, col]
        shift_x = (col - 1) * box_size
        shift_y = (1 - row) * box_size

        for j in range(10):
            p0 = x0_np[j][:2]
            p1 = x1_np[j][:2]
            pt = x_t_np[j][:2]
            c = colors[j]

            p0_s = p0 + np.array([shift_x, shift_y])
            p1_s = p1 + np.array([shift_x, shift_y])
            pt_s = pt + np.array([shift_x, shift_y])

            ax.scatter(p0_s[0], p0_s[1], c=[c], s=60, marker='o', edgecolors='black', linewidths=0.5, zorder=3)
            ax.scatter(pt_s[0], pt_s[1], c=[c], s=80, marker='x', linewidths=2, zorder=3)
            ax.scatter(p1_s[0], p1_s[1], c=[c], s=60, marker='^', edgecolors='black', linewidths=0.5, zorder=3)

        ax.set_xlim((col - 1) * box_size, col * box_size)
        ax.set_ylim((1 - row) * box_size, (2 - row) * box_size)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])

        if row == 1 and col == 1:
            ax.patch.set_facecolor('#f0f0f0')

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color('black')

ax_center = axes[1, 1]

for j in range(10):
    p0 = x0_np[j][:2]
    p1 = x1_np[j][:2]
    c = colors[j]

    delta = p1 - p0
    delta = delta - np.round(delta / box_size) * box_size
    p1_unwrapped = p0 + delta
    p0_from_p1 = p1 - delta

    ax_center.plot([p0[0], p1_unwrapped[0]], [p0[1], p1_unwrapped[1]],
                   c=c, alpha=0.5, linewidth=2, zorder=2, clip_on=False)
    ax_center.plot([p1[0], p0_from_p1[0]], [p1[1], p0_from_p1[1]],
                   c=c, alpha=0.5, linewidth=2, zorder=2, clip_on=False)

axes[0, 0].scatter([], [], c='gray', marker='o', s=40, label='x0')
axes[0, 0].scatter([], [], c='gray', marker='x', s=40, label='x_t')
axes[0, 0].scatter([], [], c='gray', marker='^', s=40, label='x1')
axes[0, 0].legend(loc='upper left')
plt.suptitle(f't = {t_val:.2f}', fontsize=14)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
plt.subplots_adjust(wspace=0, hspace=0)
box_size = 1.0
colors = cm.tab10(np.linspace(0, 1, 10))

for row in range(3):
    for col in range(3):
        ax = axes[row, col]
        shift_x = (col - 1) * box_size
        shift_y = (1 - row) * box_size

        for j in range(10):
            p0 = x0_np[j][:2]
            p1 = x1_np[j][:2]
            c = colors[j]

            p0_s = p0 + np.array([shift_x, shift_y])
            p1_s = p1 + np.array([shift_x, shift_y])

            ax.scatter(p0_s[0], p0_s[1], c=[c], s=60, marker='o', edgecolors='black', linewidths=0.5, zorder=3)
            ax.scatter(p1_s[0], p1_s[1], c=[c], s=60, marker='^', edgecolors='black', linewidths=0.5, zorder=3)

        ax.set_xlim((col - 1) * box_size, col * box_size)
        ax.set_ylim((1 - row) * box_size, (2 - row) * box_size)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])

        if row == 1 and col == 1:
            ax.patch.set_facecolor('#f0f0f0')

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color('black')

ax_center = axes[1, 1]

for j in range(10):
    p0 = x0_np[j][:2]
    p1 = x1_np[j][:2]
    c = colors[j]

    delta = p1 - p0
    delta = delta - np.round(delta / box_size) * box_size
    p1_unwrapped = p0 + delta
    p0_from_p1 = p1 - delta

    ax_center.plot([p0[0], p1_unwrapped[0]], [p0[1], p1_unwrapped[1]],
                   c=c, alpha=0.5, linewidth=2, zorder=2, clip_on=False)
    ax_center.plot([p1[0], p0_from_p1[0]], [p1[1], p0_from_p1[1]],
                   c=c, alpha=0.5, linewidth=2, zorder=2, clip_on=False)

for j in range(10):
    p0 = x0_np[j][:2]
    vel = u_t_np[j][:2]
    c = colors[j]

    ax_center.annotate('', xy=(p0[0] + vel[0], p0[1] + vel[1]), xytext=(p0[0], p0[1]),
                       arrowprops=dict(arrowstyle='->', color=c, lw=2.5),
                       zorder=5, annotation_clip=False)

axes[0, 0].scatter([], [], c='gray', marker='o', s=40, label='x0')
axes[0, 0].scatter([], [], c='gray', marker='^', s=40, label='x1')
axes[0, 0].legend(loc='upper left')
plt.suptitle(f't = {t_val:.2f}', fontsize=14)
plt.show()

In [ ]:
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

def ot_pairing(x0, x1, box_size=1.0):
    x0_np = x0.detach().clone().squeeze(0).cpu().numpy()
    x1_np = x1.detach().clone().squeeze(0).cpu().numpy()
    
    diff = x0_np[:, None, :] - x1_np[None, :, :]
    diff = diff - np.round(diff / box_size) * box_size
    cost = np.sqrt((diff**2).sum(axis=-1))
    
    row_idx, col_idx = linear_sum_assignment(cost)
    
    return col_idx

In [ ]:
perm = ot_pairing(x0, x1)
x1_paired = x1.squeeze(0)[perm].unsqueeze(0)

# Compute x_t using YOUR function
x_t_paired = calc_xt(x0, x1_paired, t)

# Convert to numpy
x1_paired_np = x1_paired.reshape(-1, 3).cpu().numpy()
x_t_paired_np = x_t_paired.reshape(-1, 3).cpu().numpy()

In [ ]:
# Plot
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
plt.subplots_adjust(wspace=0, hspace=0)
box_size = 1.0
colors = cm.tab10(np.linspace(0, 1, 10))

for row in range(3):
    for col in range(3):
        ax = axes[row, col]
        shift_x = (col - 1) * box_size
        shift_y = (1 - row) * box_size

        for j in range(10):
            p0 = x0_np[j][:2]
            p1 = x1_paired_np[j][:2]
            pt = x_t_paired_np[j][:2]
            c = colors[j]

            p0_s = p0 + np.array([shift_x, shift_y])
            p1_s = p1 + np.array([shift_x, shift_y])
            pt_s = pt + np.array([shift_x, shift_y])

            ax.scatter(p0_s[0], p0_s[1], c=[c], s=60, marker='o', edgecolors='black', linewidths=0.5, zorder=3)
            ax.scatter(pt_s[0], pt_s[1], c=[c], s=80, marker='x', linewidths=2, zorder=3)
            ax.scatter(p1_s[0], p1_s[1], c=[c], s=60, marker='^', edgecolors='black', linewidths=0.5, zorder=3)

        ax.set_xlim((col - 1) * box_size, col * box_size)
        ax.set_ylim((1 - row) * box_size, (2 - row) * box_size)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])

        if row == 1 and col == 1:
            ax.patch.set_facecolor('#f0f0f0')

        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
            spine.set_color('black')

ax_center = axes[1, 1]

for j in range(10):
    p0 = x0_np[j][:2]
    p1 = x1_paired_np[j][:2]
    c = colors[j]

    delta = p1 - p0
    delta = delta - np.round(delta / box_size) * box_size
    p1_unwrapped = p0 + delta
    p0_from_p1 = p1 - delta

    ax_center.plot([p0[0], p1_unwrapped[0]], [p0[1], p1_unwrapped[1]],
                   c=c, alpha=0.5, linewidth=2, zorder=2, clip_on=False)
    ax_center.plot([p1[0], p0_from_p1[0]], [p1[1], p0_from_p1[1]],
                   c=c, alpha=0.5, linewidth=2, zorder=2, clip_on=False)

axes[0, 0].scatter([], [], c='gray', marker='o', s=40, label='x0')
axes[0, 0].scatter([], [], c='gray', marker='x', s=40, label='x_t')
axes[0, 0].scatter([], [], c='gray', marker='^', s=40, label='x1')
axes[0, 0].legend(loc='upper left')
plt.suptitle(f'OT pairing, t = {t_val:.2f}', fontsize=14)
plt.show()

In [ ]:
idx = torch.randperm(5000)[:256]
sample = next(iter(train_loader))
mass_total = sample.mass.view(-1, 5000)

mass = mass_total[:, idx].view(-1, 1)

print(mass.shape)

In [ ]:
def minibatch_ot(x0, x1): 
    x1_ot = x1.clone()
    for b in range(x0.shape[0]):
        cost = torch.cdist(x0[b], x1[b], p=2).pow(2)
        _, col_ind = linear_sum_assignment(cost.detach().cpu().numpy())
        x1_ot[b] = x1[b, col_ind]
    
    return x1_ot

In [ ]:
x0 = torch.rand(16, 256, 3)
x1 = torch.rand(16, 256, 3)

x1_matched = minibatch_ot(x0, x1)

diff_unpaired = []
diff_paired = []

for i in range(x1.shape[0]):
    x0_graph = x0[i]
    x1_graph = x1[i]
    x1_matched_graph = x1_matched[i]

    print(f'''
    Mean absolute distance non-OT pair: {np.abs(x0_graph-x1_graph).mean()}
    Mean absolute distance OT pair: {np.abs(x0_graph-x1_matched_graph).mean()}
    ''')